# Phase 06 — Data Cleaning & Quality Filtering
## CodeMentor-LLM
Cleaning and filtering the formatted CodeAlpaca-20K dataset
to remove low quality samples before training.

## Cleaning Steps
1. Remove duplicate rows
2. Remove null or empty fields
3. Filter by token length (remove under 10 or over 2048 tokens)
4. Remove very low quality samples (single word responses)
5. Log removal counts at each step

# Libraries

In [ ]:
!pip install -q transformers==4.49.0 datasets==3.3.2 pandas==2.2.3

#  Login to HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

#  Load formatted dataset from HuggingFace Hub

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load formatted dataset
print("Loading formatted dataset...")
dataset = load_dataset("Abdulmoiz123/codementor-llm-formatted")
df = pd.DataFrame(dataset["train"])

print(f"Dataset loaded successfully")
print(f"Total samples: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst sample:")
print(df["text"][0])

# Track original count and check for nulls

In [ ]:
# Track counts at each step
cleaning_log = {}
cleaning_log["original"] = len(df)

print(f"Original samples: {cleaning_log['original']}")

# Check for null values
print(f"\nNull values:")
print((df.isnull().sum()))

# Check for empty strings
empty_count = (df["text"] == "").sum()
print(f"\nEmpty text fields: {empty_count}")

#  Remove duplicates

In [ ]:
# Remove duplicates
df_deduped = df.drop_duplicates(subset=["text"])
cleaning_log["after_dedup"] = len(df_deduped)
removed_dedup = cleaning_log["original"] - cleaning_log["after_dedup"]

print(f"Samples after deduplication: {cleaning_log['after_dedup']}")
print(f"Duplicates removed: {removed_dedup}")

# Filter by token length

In [ ]:
from transformers import AutoTokenizer

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Calculate token lengths
print("Calculating token lengths...")
df_deduped["token_length"] = df_deduped["text"].apply(
    lambda x: len(tokenizer(x)["input_ids"])
)

# Filter by token length
df_filtered = df_deduped[
    (df_deduped["token_length"] >= 10) &
    (df_deduped["token_length"] <= 2048)
]

cleaning_log["after_token_filter"] = len(df_filtered)
removed_tokens = cleaning_log["after_dedup"] - cleaning_log["after_token_filter"]

print(f"\nToken length stats:")
print(f"Min : {df_deduped['token_length'].min()}")
print(f"Max : {df_deduped['token_length'].max()}")
print(f"Mean: {df_deduped['token_length'].mean():.2f}")
print(f"\nSamples after token filter: {cleaning_log['after_token_filter']}")
print(f"Samples removed: {removed_tokens}")

# Remove low quality samples

In [ ]:
# Rule 1 — output shorter than 3 words is low quality

def is_low_quality(text):
    # Extract assistant response from formatted text
    if "<|start_header_id|>assistant<|end_header_id|>" in text:
        response = text.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
        response = response.replace("<|eot_id|>", "").strip()
        # Check if response is less than 3 words
        if len(response.split()) < 3:
            return True
    return False

# Apply filter
df_filtered["is_low_quality"] = df_filtered["text"].apply(is_low_quality)
low_quality_count = df_filtered["is_low_quality"].sum()

df_clean = df_filtered[df_filtered["is_low_quality"] == False].drop(
    columns=["is_low_quality", "token_length"]
)

cleaning_log["after_quality_filter"] = len(df_clean)
removed_quality = cleaning_log["after_token_filter"] - cleaning_log["after_quality_filter"]

print(f"Low quality samples found: {low_quality_count}")
print(f"Samples after quality filter: {cleaning_log['after_quality_filter']}")
print(f"Samples removed: {removed_quality}")

#  Print full cleaning log

In [ ]:
# Print full cleaning summary
print("CLEANING SUMMARY")
print("=" * 10)
print(f"Original samples        : {cleaning_log['original']}")
print(f"After deduplication     : {cleaning_log['after_dedup']}")
print(f"After token filter      : {cleaning_log['after_token_filter']}")
print(f"After quality filter    : {cleaning_log['after_quality_filter']}")
print(f"\nTotal removed           : {cleaning_log['original'] - cleaning_log['after_quality_filter']}")
print(f"Total remaining         : {cleaning_log['after_quality_filter']}")
print(f"Retention rate          : {cleaning_log['after_quality_filter'] / cleaning_log['original'] * 100:.2f}%")

# Save cleaned dataset to HuggingFace Hub

In [ ]:
from datasets import Dataset

# Convert to HuggingFace Dataset
clean_dataset = Dataset.from_pandas(df_clean.reset_index(drop=True))

print(f"Clean dataset size: {len(clean_dataset)}")
print(f"Columns: {clean_dataset.column_names}")

# Push to HF Hub
print("\nPushing cleaned dataset to HF Hub...")
clean_dataset.push_to_hub("Abdulmoiz123/codementor-llm-cleaned")
print("Cleaned dataset pushed successfully")

# Save cleaned dataset locally

In [ ]:
import json

# Save cleaned dataset as JSONL
with open("cleaned_dataset.jsonl", "w") as f:
    for sample in clean_dataset:
        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(clean_dataset)} samples to cleaned_dataset.jsonl")